In [ ]:
import numpy as np
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import gc
from optimizer import OneBitOptimizer

In [52]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

In [53]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.layers(x)

model = mlp()
criterion = nn.CrossEntropyLoss()
optimizer = OneBitOptimizer(model.parameters())
gc.collect()

21

In [54]:
epochs = 10
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [55]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    
    gc.collect()

Epoch [1/10]
Training loss 4.422976988474528
Training accuracy 10.205 %
Testing loss 2.310795283317566
Testing accuracy 8.39 %
Epoch [2/10]
Training loss 2.313554880777995
Training accuracy 8.233333333333333 %
Testing loss 2.314300560951233
Testing accuracy 8.66 %
Epoch [3/10]
Training loss 2.3158331873575846
Training accuracy 8.553333333333333 %
Testing loss 2.31737380027771
Testing accuracy 9.31 %
Epoch [4/10]
Training loss 2.3153588850657143
Training accuracy 9.163333333333334 %
Testing loss 2.3120569705963137
Testing accuracy 9.76 %
Epoch [5/10]
Training loss 2.314016328684489
Training accuracy 8.753333333333334 %
Testing loss 2.314441442489624
Testing accuracy 8.39 %
Epoch [6/10]
Training loss 2.311161598841349
Training accuracy 8.576666666666666 %
Testing loss 2.3105908632278442
Testing accuracy 9.12 %
Epoch [7/10]
Training loss 2.3093344997406007
Training accuracy 9.371666666666666 %
Testing loss 2.3111178874969482
Testing accuracy 9.74 %
Epoch [8/10]
Training loss 2.30832436587